# Word Similarity Using Word2Vec Models
### CBOW vs Skip-gram — Micro project (22.10.26 to 25.10.26)

**Name:** Praveen Kumar
**Registration No.:** 231FA04G42

**Main question:** Do CBOW and Skip-gram learn similar relationships between words?

**What this notebook does**

| Step | Content |
|---|---|
| 1 | The idea: how CBOW and Skip-gram differ (with the example sentence) |
| 2 | Corpus: load, clean, tokenize, describe |
| 3 | Choose five frequent words |
| 4 | Model settings and training (same settings, only `sg` differs) |
| 5 | Comparison table: top 5 similar words from both models |
| 6 | Extra evidence: overlap, agreement, human-judgement test |
| 7 | Explanation of the results and conclusion |

## 1. The idea: CBOW vs Skip-gram

Both models learn a vector (a list of numbers) for every word. Words used in similar
contexts get similar vectors. They learn it by playing a small prediction game:

* **CBOW** (Continuous Bag of Words): the **surrounding words** predict the **centre word**.
* **Skip-gram**: the **centre word** predicts each **surrounding word**.

Example sentence from the assignment: *"Students learn programming in college."*
The code below shows the training examples each model gets for the centre word **programming**
(window = 2 words on each side).

In [1]:
sentence = ["students", "learn", "programming", "in", "college"]
window = 2

def training_examples(sentence, centre_index, window):
    lo, hi = max(0, centre_index - window), min(len(sentence), centre_index + window + 1)
    context = [sentence[j] for j in range(lo, hi) if j != centre_index]
    target = sentence[centre_index]
    return context, target

context, target = training_examples(sentence, 2, window)

print("CBOW      : context", context, "-> predict", repr(target))
print("Skip-gram :", end=" ")
print(*[f"{target!r} -> {c!r}" for c in context], sep="  |  ")

CBOW      : context ['students', 'learn', 'in', 'college'] -> predict 'programming'
Skip-gram : 'programming' -> 'students'  |  'programming' -> 'learn'  |  'programming' -> 'in'  |  'programming' -> 'college'


CBOW makes **one** training example per centre word (many words in, one word out).
Skip-gram makes **one example per context word**, so it does more work per sentence
(that is why it trains more slowly) and it gives every context word its own chance to learn.

## 2. Setup

In [2]:
import sys
from collections import Counter
from itertools import islice
from pathlib import Path

import gensim, numpy, pandas as pd, sklearn

# Make the project's `src` folder importable (works from /notebooks or from the project root)
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src import compare, config, data, preprocess, train

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

print("python  :", sys.version.split()[0])
print("gensim  :", gensim.__version__)
print("numpy   :", numpy.__version__)
print("pandas  :", pd.__version__)
print("sklearn :", sklearn.__version__)

python  : 3.13.16
gensim  : 4.4.0
numpy   : 2.5.4
pandas  : 3.0.6
sklearn : 1.9.1


## 3. Corpus

**Source:** the **Brown Corpus** (Francis & Kucera). It contains 500 text samples of
American English from 15 genres (news, religion, hobbies, government documents,
scientific writing, fiction, humour and more). Its README allows redistribution, so the
zip file is stored in this repository (`data/raw/brown.zip`).

Brown is a good choice here because it is clean, mixed-topic and has tens of thousands of sentences.
Each line of the original files is one sentence, and every word carries a part-of-speech tag
(for example `The/at`). We keep only the word.

### 3.1 Raw sentences and cleaning

In [3]:
raw = list(islice(data.iter_raw_sentences(), 3))

for file_name, words in raw[:2]:
    print("RAW    :", " ".join(words))
    print("CLEAN  :", preprocess.clean_sentence(words))
    print()

RAW    : The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .
CLEAN  : ['fulton', 'county', 'grand', 'jury', 'said', 'friday', 'investigation', 'atlanta', 'recent', 'primary', 'election', 'produced', 'evidence', 'irregularities', 'took', 'place']

RAW    : The jury further said in term-end presentments that the City Executive Committee , which had over-all charge of the election , `` deserves the praise and thanks of the City of Atlanta '' for the manner in which the election was conducted .
CLEAN  : ['jury', 'said', 'term', 'end', 'presentments', 'city', 'executive', 'committee', 'charge', 'election', 'deserves', 'praise', 'thanks', 'city', 'atlanta', 'manner', 'election', 'conducted']



**Cleaning rules** (applied to every sentence)

1. lowercase every word
2. keep letters only (numbers and punctuation are removed; hyphens and apostrophes split a word)
3. remove words shorter than 3 letters
4. remove English stopwords such as *the, of, and* (scikit-learn list)
5. remove sentences that have fewer than 4 words left

In [4]:
sentences, stats = preprocess.build_corpus()
preprocess.save_sentences(sentences)   # data/processed/brown_sentences.txt
preprocess.save_stats(stats)           # results/corpus_stats.json

genres = stats.pop("sentences_per_genre_letter")
pd.Series(stats, name="value").to_frame()

,value
source,"Brown Corpus (Francis & Kucera, 1964/1979)"
text_samples,500
raw_sentences,57340
raw_tokens_incl_punctuation,1161192
sentences_kept,44509
tokens_after_cleaning,448314
vocabulary_size,40209
vocabulary_min_count_5,12982
avg_sentence_length,10.07


In [5]:
print("Example cleaned sentences (the input of Word2Vec):")
for s in sentences[100:104]:
    print(" ", s)

Example cleaned sentences (the input of Word2Vec):
  ['cox', 'argued', 'probably', 'unconstitutional', 'said', 'impair', 'contracts']
  ['complained', 'notice', 'given', 'hearing', 'introduced', 'monday']
  ['senators', 'unanimously', 'approved', 'thursday', 'sen', 'george', 'parkhouse', 'dallas', 'authorizing', 'establishment', 'day', 'schools', 'deaf', 'dallas', 'largest', 'counties']
  ['designed', 'provide', 'special', 'schooling', 'deaf', 'students', 'scholastic', 'age', 'reduced', 'cost', 'state']


## 4. Choose five frequent words

The assignment asks for five words that occur frequently in the corpus.
First look at the most frequent words of the cleaned corpus.

In [6]:
counts = Counter(t for s in sentences for t in s)
top = pd.DataFrame(counts.most_common(100), columns=["word", "count"])
top.index = top.index + 1
top.index.name = "rank"
top.head(30).T

rank,1,2,3,4,5,6,7,8,9,10,...,21,22,23,24,25,26,27,28,29,30
word,said,new,time,like,man,years,state,did,way,long,...,old,life,great,states,used,american,use,came,house,high
count,1605,1597,1579,1266,1263,919,890,885,848,830,...,718,695,650,604,597,594,580,575,573,572


I selected five **nouns from different topic areas**, all inside the 100 most frequent words:

* **school** (education), **government** (politics), **water** (nature), **war** (conflict), **house** (everyday life, and also an ambiguous word).

Using different topics tests whether the two models agree *in general*, not only in one subject.

In [7]:
words = config.SELECTED_WORDS
rank = {w: i + 1 for i, (w, _) in enumerate(counts.most_common())}
pd.DataFrame({"word": words,
              "count in corpus": [counts[w] for w in words],
              "frequency rank": [rank[w] for w in words]})

,word,count in corpus,frequency rank
0,school,502,38
1,government,432,51
2,water,442,49
3,war,482,40
4,house,573,29


## 5. Model settings and training

For a **fair comparison** every setting is identical. **Only `sg` changes**
(`sg=0` → CBOW, `sg=1` → Skip-gram).

In [8]:
pd.DataFrame(config.W2V_PARAMS, index=["value"]).T.rename_axis("setting")

,value
setting,
vector_size,100.000
window,5.000
min_count,5.000
epochs,20.000
negative,5.000
sample,0.001
workers,1.000
seed,42.000


| Setting | Meaning |
|---|---|
| `vector_size=100` | each word becomes 100 numbers |
| `window=5` | up to 5 words on each side are context |
| `min_count=5` | words that appear fewer than 5 times are ignored (too little evidence) |
| `epochs=20` | the model reads the whole corpus 20 times |
| `negative=5` | negative sampling: 5 wrong words per example make training fast |
| `sample=1e-3` | very frequent words are randomly skipped sometimes |
| `workers=1`, `seed=42` | one worker thread and a fixed seed so the results can be repeated |

In [9]:
cbow, skip, timings = train.train_models(sentences)
print("Training time:", timings)

Training time: {'cbow_seconds': 13.7, 'skipgram_seconds': 36.4}


In [10]:
# Proof that the two models differ ONLY in sg:
settings = pd.DataFrame({"CBOW": train.model_settings(cbow),
                         "Skip-gram": train.model_settings(skip)})
display(settings)
different = settings.index[settings["CBOW"] != settings["Skip-gram"]].tolist()
print("Settings that differ between the two models:", different)
assert different == ["sg"]

,CBOW,Skip-gram
sg,0.000,1.000
vector_size,100.000,100.000
window,5.000,5.000
min_count,5.000,5.000
epochs,20.000,20.000
negative,5.000,5.000
sample,0.001,0.001
workers,1.000,1.000
seed,42.000,42.000
vocabulary,12982.000,12982.000


Settings that differ between the two models: ['sg']


## 6. Comparison: top 5 similar words
Similarity is the **cosine similarity** between word vectors (1 = same direction, 0 = unrelated).

In [11]:
table = compare.comparison_table(cbow, skip, config.SELECTED_WORDS)
config.RESULTS_DIR.mkdir(exist_ok=True)
table.to_csv(config.RESULTS_DIR / "comparison_table.csv", index=False)
table

,Word,Frequency,CBOW top 5,Skip-gram top 5,Common words,Overlap
0,school,502,"schools, teacher, divinity, graduate, structured","retarded, desegregation, elementary, graduates, mentally",-,0/5
1,government,432,"constitution, foreign, governments, coalition, abolish","belgian, governments, balafrej, currency, yugoslavia",governments,1/5
2,water,442,"ice, hot, salt, cooled, soil","rinse, bucket, frontage, boil, reservoir",-,0/5
3,war,482,"renowned, battle, citizen, civil, wars","civil, fronts, korean, custer, propaganda",civil,1/5
4,house,573,"sox, hotel, mob, kent, apartment","siding, coffin, barber, livery, fireplace",-,0/5


In [12]:
scores = compare.neighbours_with_scores(cbow, skip, config.SELECTED_WORDS)
scores.to_csv(config.RESULTS_DIR / "neighbours_with_scores.csv", index=False)

for word in config.SELECTED_WORDS:
    print(word.upper())
    for model in ("CBOW", "Skip-gram"):
        part = scores[(scores.Word == word) & (scores.Model == model)]
        print(f"  {model:<10}", ",  ".join(f"{n} ({c})" for n, c in zip(part.Neighbour, part.Cosine)))
    print()

SCHOOL
  CBOW       schools (0.848),  teacher (0.769),  divinity (0.725),  graduate (0.719),  structured (0.718)
  Skip-gram  retarded (0.636),  desegregation (0.631),  elementary (0.628),  graduates (0.617),  mentally (0.608)

GOVERNMENT
  CBOW       constitution (0.844),  foreign (0.837),  governments (0.829),  coalition (0.819),  abolish (0.817)
  Skip-gram  belgian (0.59),  governments (0.587),  balafrej (0.58),  currency (0.58),  yugoslavia (0.569)

WATER
  CBOW       ice (0.723),  hot (0.716),  salt (0.711),  cooled (0.709),  soil (0.704)
  Skip-gram  rinse (0.591),  bucket (0.574),  frontage (0.568),  boil (0.565),  reservoir (0.559)

WAR
  CBOW       renowned (0.7),  battle (0.662),  citizen (0.645),  civil (0.638),  wars (0.62)
  Skip-gram  civil (0.622),  fronts (0.605),  korean (0.519),  custer (0.517),  propaganda (0.509)

HOUSE
  CBOW       sox (0.685),  hotel (0.578),  mob (0.545),  kent (0.52),  apartment (0.515)
  Skip-gram  siding (0.505),  coffin (0.492),  barber (0.4

## 7. Extra evidence

Five words are a small sample, so three more checks are added.

### 7.1 Overlap over the 200 most frequent words
For each word, how many of the 5 neighbours are the same in both models?

In [13]:
overlap = compare.overlap_summary(cbow, skip)
print({k: v for k, v in overlap.items() if k != "distribution"})
pd.Series(overlap["distribution"], name="number of words").rename_axis("common neighbours (out of 5)").to_frame().T

{'words_checked': 200, 'k': 5, 'mean_overlap': 0.63, 'share_with_at_least_1_common': 0.41, 'share_with_3_or_more_common': 0.045, 'chance_level_mean_overlap': 0.0019}


common neighbours (out of 5),0,1,2,3,4,5
number of words,118,47,26,9,0,0


*Chance level:* if both models picked 5 random words from the vocabulary, the expected overlap would be
about 0.002. Any clearly larger value means the two models really share information.

### 7.2 Do the two models rank the *whole vocabulary* in a similar order?
For one word, both models give a similarity score to **every** other word. The Spearman correlation
compares the two rankings (1 = identical order, 0 = unrelated).

In [14]:
agree = pd.Series({w: compare.similarity_agreement(cbow, skip, w) for w in config.SELECTED_WORDS},
                  name="Spearman correlation").round(3)
agree.to_frame()

,Spearman correlation
school,0.785
government,0.809
water,0.715
war,0.729
house,0.785


### 7.3 What kind of neighbours does each model prefer?

In [15]:
profile = pd.DataFrame({"CBOW": compare.neighbour_profile(cbow),
                        "Skip-gram": compare.neighbour_profile(skip)})
profile

,CBOW,Skip-gram
median_neighbour_count_in_corpus,52.000,9.000
mean_cosine_of_top5,0.718,0.521


### 7.4 Agreement with human judgement (WordSim-353)
WordSim-353 is a standard list of 353 word pairs that people rated for similarity
(e.g. *tiger – cat*). A higher Spearman value means the model agrees more with people.
Pairs that contain a word missing from our vocabulary are skipped.

In [16]:
pd.DataFrame({"CBOW": compare.wordsim_evaluation(cbow),
              "Skip-gram": compare.wordsim_evaluation(skip)})

,CBOW,Skip-gram
spearman,0.296,0.446
pearson,0.301,0.445
pairs_not_covered_percent,24.600,24.600


In [17]:
import json
extra = {
    "overlap_top200": overlap,
    "agreement_selected_words": agree.to_dict(),
    "neighbour_profile_top200": {"cbow": compare.neighbour_profile(cbow),
                                 "skipgram": compare.neighbour_profile(skip)},
    "wordsim353": {"cbow": compare.wordsim_evaluation(cbow),
                   "skipgram": compare.wordsim_evaluation(skip)},
}
_ = (config.RESULTS_DIR / "extra_evidence.json").write_text(json.dumps(extra, indent=2))

## 8. Explanation of the results

*(The numbers below come from the run saved in this notebook. The run is repeatable because
the seed and the number of workers are fixed.)*

**Answer to the main question:** the two models learn **related but not identical** word relationships.
They agree on the broad picture, but they often disagree about which words are the *closest five*.

**1. The top-5 lists are mostly different.**
For the five selected words only 2 of the 25 neighbour slots are shared (*governments* for
"government" and *civil* for "war"). Over the 200 most frequent words the average overlap is
0.63 out of 5. That is far above the chance level (0.002), but still small.

**2. The overall ordering of the vocabulary is quite similar.**
Looking at all words, not only the top five, the Spearman correlation for the selected words is
between 0.72 and 0.81. So both models place the same kinds of words "near" and "far" from a given word;
they differ mostly in which close words come first.

**3. Which relationships look meaningful?**

| Word | CBOW (meaningful) | Skip-gram (meaningful) | Weak or noisy |
|---|---|---|---|
| school | schools, teacher, graduate | elementary, graduates, desegregation | CBOW: structured, divinity |
| government | constitution, governments, coalition, foreign | governments, currency | Skip-gram: balafrej (a name seen very few times) |
| water | ice, hot, salt, cooled, soil | rinse, boil, bucket, reservoir | Skip-gram: frontage |
| war | battle, wars, civil | civil, korean, propaganda, fronts | CBOW: renowned |
| house | hotel, apartment | fireplace, coffin, barber, livery | CBOW: sox, kent (noise, "house" is ambiguous) |

**4. The two models have different "styles".**
CBOW's neighbours are **common words** (median corpus count 52) with higher cosine scores (0.72 on average).
Skip-gram's neighbours are **rarer, more specific words** (median count 9) with lower scores (0.52).
Skip-gram gives more specific topic words (*desegregation*, *reservoir*, *propaganda*), but because it
also lifts words that occur only a few times, some of its neighbours are names or accidents of the corpus.
(Cosine scores of two different models are only a rough guide and should not be compared too strictly.)

**5. Agreement with human judgement.**
On WordSim-353 (about 75% of the pairs are covered by our vocabulary), Skip-gram reaches a Spearman value of
0.446 and CBOW 0.296. In this experiment, Skip-gram's similarities are closer to human judgement.

**Conclusion.** CBOW and Skip-gram both learn meaningful topical relations
(school – elementary/graduate, war – battle/civil, water – boil/ice), but they produce different top-5 lists:
CBOW prefers frequent, general neighbours, and Skip-gram prefers rarer, more specific neighbours and agreed
better with human ratings.

**Limitations.** The corpus is small (about 448,000 words after cleaning), so vectors of rarer words are noisy.
Only one corpus, one seed and one set of settings were tested, stopwords were removed (which changes
what counts as "next to" a word), and words with several meanings (*house*) mix their contexts.
A larger corpus or several seeds would make the conclusions stronger.

## 9. How to reproduce

```bash
python -m venv .venv
.venv\Scripts\activate          # Windows  (Linux/Mac: source .venv/bin/activate)
pip install -r requirements.txt
jupyter notebook notebooks/word2vec_comparison.ipynb
# or run everything without the notebook:
python main.py
```